In [ ]:
from src.utils.config import popularity, load_env_file
from src.utils.files import read_file

from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor

import pandas as pd
import numpy as np
import optuna
import math
import warnings
import xgboost as xgb

from umap import UMAP

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import matplotlib.pyplot as plt

import pprint
import optuna.visualization as vis

In [ ]:
load_env_file()
df = read_file(popularity, minio={"minio_write": False, "minio_read": False})

In [ ]:
def get_metrics(y_test, y_pred):
    '''
    Dados el output predecido del modelo y los datos reales, se construyen
    las métricas necesarias para medir el rendimiento de un modelo de REGRESIÓN.
    '''
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print(f'MAE (Error Absoluto Medio): {mae:.4f}')
    print(f'RMSE (Raíz del Error Cuadrático Medio): {rmse:.4f}')
    print(f'R2 Score (Coef. de Determinación): {r2:.4f}')

    return {'mae': mae, 'rmse': rmse, 'r2': r2}

In [ ]:
def _preprocess(df, image_col='v_clip', use_images=True):
    images_cols = ['v_clip', 'v_convnext', 'v_resnet']

    assert image_col in images_cols, "Seleccione una columna de imágenes válida"

    df_clean = df.copy()
    
    # Encuentra todas las columnas que tengan "video_statistics" en su nombre
    video_cols = [col for col in df_clean.columns if 'video_statistics' in col]

    # Seleccionamos las variables de entrada útiles
    target_col = df_clean['recomendaciones_totales']
    erase_columns = ['id', 'name', 'price_range']
    erase_columns.extend(video_cols)
    images_cols.remove(image_col)
    erase_columns.extend(images_cols)
    df_clean = df_clean.drop(columns=erase_columns, errors='ignore')

    if use_images:
        # Vamos a preparar el DataFrame para que scikit-learn haga el PCA mas adelante
        # Forzamos a que todos los valores de la columna de los embeddings sean iterables
        zero_vector = np.zeros(512)
        df_clean[image_col] = df_clean[image_col].apply(
            lambda x: x if isinstance(x, (list, np.ndarray)) else zero_vector
        )
        
        img_df = pd.DataFrame(df_clean[image_col].tolist(), index=df_clean.index)
        img_df.columns = [f'{image_col}_{i}' for i in range(img_df.shape[1])]
        df_clean = pd.concat([df_clean.drop(columns=[image_col]), img_df], axis=1)
    else:
        # Ya la columna de los embeddings originales no es necesaria
        df_clean = df_clean.drop(columns=[image_col], errors='ignore')

    obj_cols = df_clean.select_dtypes(include=['object', 'str']).columns
    for col in obj_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    # Solo nos quedamos con columnas numéricas y rellenamos nulos
    df_clean = df_clean.select_dtypes(include=[np.number])
    df_clean = df_clean.fillna(0)

    df_clean['recomendaciones_totales'] = target_col

    return df_clean

Vamos a analizar las variables para saber que transformaciones se deben hacer con ellas

In [ ]:
target_name = 'recomendaciones_totales' 
df_analisis_variables = _preprocess(df, image_col='v_clip', use_images=False)

X_analisis = df_analisis_variables.drop(columns=[target_name])
y_analisis = df_analisis_variables[target_name]

print("ASIMETRÍA DE LA VARIABLE RESPUESTA")
print("-" * 40)
sesgo = y_analisis.skew()
problematicas = sesgo[abs(sesgo) > 1]

if len(problematicas) > 0:
    print("La variable respuesta está muy sesgada (valor > 1 o < -1):")
    print(problematicas)
else:
    print("La variable respuesta no presenta un sesgo grave.")

print("\nASIMETRÍA DE LAS VARIABLES REGRESORAS")
print("-" * 40)
sesgo = X_analisis.skew().sort_values(ascending=False)
problematicas = sesgo[abs(sesgo) > 1]

if len(problematicas) > 0:
    print("Las siguientes variables están muy sesgadas (valor > 1 o < -1):")
    print(problematicas)
else:
    print("Ninguna variable presenta un sesgo grave.")

print("\nHISTOGRAMAS DE LAS VARIABLES")
print("-" * 40)

# Calcular la cuadrícula para los subplots
columnas = df_analisis_variables.columns.tolist()
n_cols = 4
n_rows = math.ceil(len(columnas) / n_cols)

# Crear la figura de subplots de Plotly
fig = make_subplots(
    rows=n_rows, 
    cols=n_cols, 
    subplot_titles=columnas,
    vertical_spacing=0.05,
    horizontal_spacing=0.05
)

# Añadir un histograma por cada columna de X
for i, col in enumerate(columnas):
    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1
    
    fig.add_trace(
        go.Histogram(
            x=df_analisis_variables[col], 
            name=col, 
            marker=dict(color='#4CAF50', line=dict(color='black', width=1))
        ),
        row=row, 
        col=col_pos
    )

# Ajustar el diseño del gráfico general
fig.update_layout(
    title_text="Distribución de las variables de entrada (X)",
    title_x=0.5,
    title_font_size=20,
    height=300 * n_rows,  # Altura dinámica dependiendo del número de variables
    showlegend=False,
    plot_bgcolor='white'
)

# Mostrar el gráfico interactivo
fig.show()

Como k-nn sufre mucho al tener muchas dimensiones, vamos a analizar cuales son las más importantes con un modelo de XGBoost

In [ ]:
# Silenciar específicamente los avisos para que la salida sea limpia
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_xgb(trial, X_t, y_t, image_col, use_images):
    '''
    Función objetivo a optimizar por Optuna para XGBoost
    '''
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBRegressor(**params)
    
    steps = []
    
    img_cols = [c for c in X_t.columns if c.startswith(f'{image_col}_')] if use_images else []
    tabular_cols = [c for c in X_t.columns if c not in img_cols]
    
    transformers_list = [
        ('tabular', 'passthrough', tabular_cols)
    ]
    
    if use_images:
        dim_reduction = trial.suggest_categorical('dim_reduction', ['pca', 'umap'])
        
        if dim_reduction == 'pca':
            reducer = PCA(n_components=10, random_state=42)
        else:
            n_neighbors = trial.suggest_int('umap_n_neighbors', 5, 50)
            min_dist = trial.suggest_float('umap_min_dist', 0.0, 0.5)
            reducer = UMAP(n_components=10, n_neighbors=n_neighbors, min_dist=min_dist, random_state=42)
            
        transformers_list.append(('reducer', reducer, img_cols))
        
    preprocessor = ColumnTransformer(transformers=transformers_list, remainder='drop')
    steps.append(('prep', preprocessor))
    steps.append(('xgb', model))
    
    pipeline = Pipeline(steps)
    
    # Transformación de la variable respuesta
    final_model = TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )
    
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(final_model, X_t, y_t, cv=cv, scoring='neg_mean_absolute_error', return_train_score=True,
                            n_jobs=-1)
    
    trial.set_user_attr("train_mae", -scores['train_score'].mean())
    
    return -scores['test_score'].mean()

combinations = [['v_clip', True]]

results = []
diccionario_importancias = {}

for c in combinations:
    print(f"\n{'='*50}")
    print(f"Evaluando combinación: {c}")
    
    df_prepared = _preprocess(df, image_col=c[0], use_images=c[1])

    X = df_prepared.drop(columns=['recomendaciones_totales'])
    y = df_prepared['recomendaciones_totales']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective_xgb(trial, X_train, y_train, c[0], c[1]), n_trials=35)

    print(f"Mejor MAE Validación: {study.best_value:.4f} | MAE Train: {study.best_trial.user_attrs['train_mae']:.4f}")
    
    best_params = study.best_params
    
    if c[1]:
        tecnica_ganadora = best_params.get('dim_reduction')
        print(f"\n---> INFO PARA EL KNN <---")
        print(f"Técnica de reducción de imágenes ganadora: {tecnica_ganadora.upper()}")
        
        if tecnica_ganadora == 'umap':
            print(f"n_neighbors óptimo para UMAP: {best_params.get('umap_n_neighbors')}")
            print(f"min_dist óptimo para UMAP: {best_params.get('umap_min_dist')}")

    xgb_params = {k: v for k, v in best_params.items() if k not in ['dim_reduction', 'umap_n_neighbors', 'umap_min_dist']}
    best_xgb = xgb.XGBRegressor(**xgb_params, random_state=42, n_jobs=-1)
    
    cols_sesgadas = ['price_overview', 'num_languages', 'total_games_by_publisher', 'total_games_by_developer']
    img_cols = [col for col in X_train.columns if col.startswith(f"{c[0]}_")] if c[1] else []
    cols_binarias = [col for col in X_train.columns if set(X_train[col].dropna().unique()).issubset({0, 1, 0.0, 1.0})]
    cols_normales = [col for col in X_train.columns if col not in cols_sesgadas + img_cols + cols_binarias]
    
    final_transformers = [
        ('sesgadas', PowerTransformer(method='yeo-johnson'), cols_sesgadas),
        ('normales', StandardScaler(), cols_normales),
        ('binarias', 'passthrough', cols_binarias)
    ]
    
    if c[1]:
        dim_reduction = best_params.get('dim_reduction', 'pca')
        if dim_reduction == 'pca':
            final_reducer = PCA(n_components=10, random_state=42)
        else:
            final_reducer = UMAP(n_components=10, n_neighbors=best_params['umap_n_neighbors'], min_dist=best_params['umap_min_dist'], random_state=42)
        final_transformers.append(('reducer', final_reducer, img_cols))
        
    final_preprocessor = ColumnTransformer(transformers=final_transformers, remainder='drop')
    final_pipeline = Pipeline([('prep', final_preprocessor), ('xgb', best_xgb)])
    final_model = TransformedTargetRegressor(regressor=final_pipeline, func=np.log1p, inverse_func=np.expm1)
    
    final_model.fit(X_train, y_train)
    final_preds = final_model.predict(X_test)
    
    metricas = get_metrics(y_test, final_preds)
    
    results.append({
        'image_col': c[0], 
        'use_images': c[1], 
        'val_mae': study.best_value,
        'train_mae': study.best_trial.user_attrs['train_mae'], 
        'test_mae': metricas['mae'],
        'test_rmse': metricas['rmse'],
        'test_r2': metricas['r2']
    })

    modelo_entrenado = final_model.regressor_['xgb']
    # Obtenemos los pesos de cada variable
    pesos = modelo_entrenado.feature_importances_
    
    # Creamos nuestra propia lista de nombres en el mismo orden
    # en el que metimos las variables en el ColumnTransformer
    mis_nombres = cols_sesgadas + cols_normales + cols_binarias
    
    # Si usamos PCA o UMAP, añadimos 10 nombres extra
    if c[1] == True:
        mis_nombres = mis_nombres + [f"img_reducida_{i}" for i in range(10)]

    df_importancias = pd.DataFrame({
        'Variable': mis_nombres,
        'Importancia': pesos
    })
    
    print("\nTOP 10 Variables más importantes")
    display(df_importancias.sort_values(by='Importancia', ascending=False).head(10))

# Imprimimos el resumen de resultados final
df_results = pd.DataFrame(results).sort_values('test_mae', ascending=True)
print("\nRESUMEN FINAL DE RESULTADOS:")
display(df_results)

In [ ]:
display(df_importancias.sort_values(by='Importancia', ascending=False).head(20).reset_index(drop=True))

Observamos que las variables mas significativas son: 
1. Family Sharing
2. Free To Play
3. yt_score
4. Steam Trading Cards
5. price_overview
6. Steam Cloud
7. num_languages
8. Steam Achievements
9. RPG
10. Online Co-op
11. Casual
12. img_reducida_0
13. Custom Volume Controls
14. Simulation
15. release_year
16. Co-op
17. Multi-player
18. Shared/Split Screen
19. Playable without Timed Input
20. total_games_by_publisher


In [ ]:
# Ocultamos las advertencias para mantener la consola limpia
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Preparamos los datos
df_prepared = _preprocess(df, use_images=True)
X = df_prepared.drop(columns=['recomendaciones_totales'])
y = df_prepared['recomendaciones_totales']

# Agrupamos la popularidad en rangos para estratificar
bins_strat = [-1, 10, 100, 1000, 10000, float('inf')]
y_binned = pd.cut(y, bins=bins_strat, labels=False)

# Dividimos los datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y_binned
)

# Variables candidatas a utilizar en el modelo
top_tabular_variables = [
    'Family Sharing', 'Free To Play', 'yt_score', 'Steam Trading Cards',
    'price_overview', 'Steam Cloud', 'num_languages', 'Steam Achievements',
    'RPG', 'Online Co-op', 'Casual', 'Custom Volume Controls',
    'Simulation', 'release_year', 'Co-op', 'Multi-player', 'Shared/Split Screen',
    'Playable without Timed Input', 'total_games_by_publisher'
]

def rmsle_scorer(y_true, y_pred):
    """
    Calcula el error RMSLE. Limita las predicciones negativas a 0 
    para evitar errores matemáticos al aplicar el logaritmo.
    """
    y_pred_safe = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_log_error(y_true, y_pred_safe))

def objective(trial, X_t, y_t):
    """
    Función objetivo para Optuna: prueba diferentes configuraciones 
    para encontrar el modelo con menor error.
    """
    img_cols = [c for c in X_t.columns if c.startswith('v_clip_')]
    tabular_cols = [c for c in top_tabular_variables if c in X_t.columns]

    # Optuna decide aleatoriamente qué variables usar en este intento
    variables_elegidas = [col for col in tabular_cols if trial.suggest_categorical(f'usar_{col}', [True, False])]
    if len(variables_elegidas) == 0:
        return float('inf') # Descartar si no elige ninguna variable

    columnas_finales = variables_elegidas + img_cols
    X_filtrada = X_t[columnas_finales]

    # Optuna prueba distintos parámetros para el algoritmo KNN
    n_neighbors = trial.suggest_int('n_neighbors', 3, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    p = trial.suggest_int('p', 1, 2)

    # Identificamos el tipo de cada variable para aplicar la transformación matemática adecuada
    todas_sesgadas = ['price_overview', 'num_languages', 'total_games_by_publisher']
    cols_sesgadas = [c for c in variables_elegidas if c in todas_sesgadas]
    cols_binarias = [c for c in variables_elegidas if set(X_t[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})]
    cols_normales = [c for c in variables_elegidas if c not in cols_sesgadas + cols_binarias]

    # Preprocesamiento: normalización y estandarización de los datos numéricos
    transformers_list = [
        ('sesgadas', PowerTransformer(method='yeo-johnson'), cols_sesgadas),
        ('normales', StandardScaler(), cols_normales),
        ('binarias', 'passthrough', cols_binarias)
    ]

    # Reducción de dimensionalidad (PCA) para comprimir los datos de las imágenes
    if len(img_cols) > 0:
        reducer = PCA(n_components=10, random_state=42)
        transformers_list.append(('reducer', reducer, img_cols))

    # Construimos el pipeline con el preprocesamiento y el modelo KNN
    preprocessor = ColumnTransformer(transformers=transformers_list, remainder='drop')
    pipeline = Pipeline([
        ('prep', preprocessor),
        ('knn', KNeighborsRegressor(n_neighbors=n_neighbors, weights=weights, p=p, n_jobs=-1))
    ])
    
    # Aplicamos logaritmo a la variable objetivo internamente para manejar su gran varianza
    final_model = TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)

    # Evaluamos el modelo usando validación cruzada (5 particiones)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(
        final_model, X_filtrada, y_t,
        cv=cv,
        scoring='neg_mean_squared_log_error',
        n_jobs=-1
    )
    return np.sqrt(-scores['test_score'].mean()) # Retorna el error medio a minimizar

# Ejecutamos la optimización realizando 300 pruebas
study = optuna.create_study(direction='minimize')
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=300)

# Extraemos y mostramos la importancia de cada parámetro probado
importancias = optuna.importance.get_param_importances(study)
df_importancias = pd.DataFrame(list(importancias.items()), columns=['Parámetro', 'Importancia (%)'])
df_importancias['Importancia (%)'] = (df_importancias['Importancia (%)'] * 100).round(2)

print("\nPARÁMETROS Y VARIABLES MÁS INFLUYENTES")
try:
    display(df_importancias.head(15))
except NameError:
    print(df_importancias.head(15))

try:
    fig = vis.plot_param_importances(study)
    fig.show()
except Exception as e:
    print(f"No se pudo mostrar el gráfico interactivo: {e}")

# CONSTRUCCIÓN DEL MODELO DEFINITIVO
# Seleccionamos la mejor configuración y las variables ganadoras encontradas por Optuna
mejores_parametros = study.best_params
img_cols = [c for c in X_train.columns if c.startswith('v_clip_')]
variables_ganadoras = [col for col in top_tabular_variables if mejores_parametros.get(f'usar_{col}') == True]

# Filtramos los datos de entrenamiento y prueba usando solo las variables ganadoras
X_train_final = X_train[variables_ganadoras + img_cols]
X_test_final  = X_test[variables_ganadoras + img_cols]

# Reconstruimos los transformadores con la lista definitiva de variables
todas_sesgadas = ['price_overview', 'num_languages', 'total_games_by_publisher']
cols_sesgadas = [c for c in variables_ganadoras if c in todas_sesgadas]
cols_binarias = [c for c in variables_ganadoras if set(X_train[c].dropna().unique()).issubset({0, 1, 0.0, 1.0})]
cols_normales = [c for c in variables_ganadoras if c not in cols_sesgadas + cols_binarias]

final_transformers = [
    ('sesgadas', PowerTransformer(method='yeo-johnson'), cols_sesgadas),
    ('normales', StandardScaler(), cols_normales),
    ('binarias', 'passthrough', cols_binarias)
]

if len(img_cols) > 0:
    final_reducer = PCA(n_components=10, random_state=42)
    final_transformers.append(('reducer', final_reducer, img_cols))

# Creamos el pipeline final con los mejores parámetros
final_pipeline = Pipeline([
    ('prep', ColumnTransformer(transformers=final_transformers, remainder='drop')),
    ('knn', KNeighborsRegressor(
        n_neighbors=mejores_parametros['n_neighbors'],
        weights=mejores_parametros['weights'],
        p=mejores_parametros['p'],
        n_jobs=-1
    ))
])

modelo_definitivo = TransformedTargetRegressor(
    regressor=final_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# Entrenamos el modelo final y generamos las predicciones sobre el conjunto de prueba
modelo_definitivo.fit(X_train_final, y_train)
predicciones = modelo_definitivo.predict(X_test_final)

# EVALUACIÓN DE RESULTADOS
print("\nMÉTRICAS GENERALES EN TEST")
try:
    get_metrics(y_test, predicciones)
except NameError:
    print(f"MAE: {mean_absolute_error(y_test, predicciones):.4f}")

rmsle_test = rmsle_scorer(y_test, predicciones)
print(f"RMSLE: {rmsle_test:.4f}")

def check_performance_by_bins(y_real, y_pred):
    """
    Evalúa el error absoluto medio (MAE) segmentando los juegos 
    en distintos rangos según su popularidad real.
    """
    results = pd.DataFrame({'Real': y_real, 'Pred': y_pred})
    bins   = [-1, 10, 100, 1000, 10000, float('inf')]
    labels = ['0-10', '10-100', '100-1k', '1k-10k', '>10k']
    results['Rango'] = pd.cut(results['Real'], bins=bins, labels=labels)

    performance = results.groupby('Rango', observed=True).apply(
        lambda x: mean_absolute_error(x['Real'], x['Pred']) if len(x) > 0 else 0
    )

    print("\nMAE POR RANGO DE POPULARIDAD")
    print(performance)

    # Gráfico de dispersión para visualizar Predicciones vs Reales (en escala logarítmica)
    plt.figure(figsize=(10, 5))
    plt.scatter(y_real, y_pred, alpha=0.4, color='blue')
    plt.plot([y_real.min(), y_real.max()], [y_real.min(), y_real.max()], 'r--')
    plt.xscale('log'); plt.yscale('log')
    plt.title('Dispersión Real vs Predicho (Escala Logarítmica)')
    plt.xlabel('Recomendaciones Reales')
    plt.ylabel('Predicciones')
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.show()

check_performance_by_bins(y_test, predicciones)

# Exportamos los hiperparámetros ganadores y las variables seleccionadas
dict_exportar = {
    "modelo": "KNN_POPULARIDAD",
    "knn_params": {
        "n_neighbors": mejores_parametros["n_neighbors"],
        "weights": mejores_parametros["weights"],
        "p": mejores_parametros["p"]
    },
    "variables_seleccionadas": variables_ganadoras
}

pprint.pprint(dict_exportar, indent=4, sort_dicts=False)